# Iterative quantum phase estimation

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do we extract the molecular ground-state energy from the prepared quantum state?

</div>

</div>

## Learning objectives

After completing this chapter, you will be able to:

- Relate a Hamiltonian eigenvalue to a time-evolution eigenphase.
- Describe how IQPE measures phase bits.
- Explain the roles of the compute register and readout ancilla.
- Relate phase bits, shots per bit, and repeated complete runs to the energy estimate.
- Run IQPE with native QDK/Chemistry tools.
- Reconstruct the total molecular energy and evaluate its error.

## Before you begin

This course requires a Python environment with the `qdk-chemistry[jupyter]` package.

`qdk-chemistry` ships compiled binaries for Linux, macOS on Apple silicon, and Windows on x86-64. This course also needs PySCF, which has no Windows build, so run it inside WSL on Windows. Run the cell below to check the current environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

## Setting up

The cell below imports the QDK/Chemistry pieces this chapter uses and quiets the solver logs.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from time import perf_counter

import numpy as np
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import AlgorithmRef, Circuit, QubitOperator
from qdk_chemistry.utils import Logger
from tutorial_map_n2_to_qubits import QubitMappingResult, run_qubit_mapping_workflow
from tutorial_prepare_trial_state import TrialStateResult, run_trial_state_workflow

DEFAULT_NUM_PHASE_BITS = 6
DEFAULT_SHOTS_PER_BIT = 3
DEFAULT_NUM_COMPLETE_RUNS = 20
DEFAULT_FIRST_SEED = 42
TARGET_ENERGY_ERROR_HARTREE = 1e-3

@dataclass
class EvolutionTimeChoice:
    r"""Intermediate and final values in reference-guided time selection.

    Attributes:
        bound_time_hartree_inverse: Initial time :math:`\pi/\lambda`, in inverse
            Hartree, whose signed phase interval contains the bounded spectrum.
        bound_reference_phase_fraction: Reference phase fraction at the bound time.
        time_hartree_inverse: Final adjusted evolution time in inverse Hartree.
        reference_phase_fraction: Reference phase fraction at the adjusted time.
        grid_phase_fraction: Selected finite-bit phase-grid fraction.
        grid_bitstring: Most-significant-bit-first representation of that fraction.
        grid_active_energy_hartree: Active-space energy reconstructed from the
            selected grid point, in Hartree.
    """

    bound_time_hartree_inverse: float
    bound_reference_phase_fraction: float
    time_hartree_inverse: float
    reference_phase_fraction: float
    grid_phase_fraction: float
    grid_bitstring: str
    grid_active_energy_hartree: float

@dataclass
class IqpeProblem:
    """Prepared Hamiltonian, trial state, controls, and iteration circuits.

    Attributes:
        mapping: Selected-space fermion-to-qubit mapping result.
        trial_state: Selected trial-state and preparation circuit.
        evolution_time: Reference-guided phase-grid/time selection.
        num_phase_bits: Number of bitwise IQPE iterations.
        shots_per_bit: Simulator executions used for each bit majority vote.
        circuit_builder_reference: Serializable nested algorithm configuration
            supplied to the phase-estimation implementation.
        iteration_circuits: Constructed circuits in controlled-power order; the
            list length equals ``num_phase_bits``.
    """

    mapping: QubitMappingResult
    trial_state: TrialStateResult
    evolution_time: EvolutionTimeChoice
    num_phase_bits: int
    shots_per_bit: int
    circuit_builder_reference: AlgorithmRef
    iteration_circuits: list[Circuit]

@dataclass
class IqpeRun:
    """Measurements, energies, and timing from one complete bitwise IQPE run.

    Attributes:
        seed: Full-state simulator seed.
        bitstring: Canonical most-significant-bit-first phase-grid label.
        phase_fraction: Measured phase fraction in ``[0, 1)``.
        active_energy_hartree: Eigenvalue returned for the qubit Hamiltonian.
        total_energy_hartree: Active energy plus the classically stored core energy.
        error_hartree: Total energy minus the selected-space CASCI reference.
        runtime_seconds: Wall-clock simulator time for this complete six-bit run.
    """

    seed: int
    bitstring: str
    phase_fraction: float
    active_energy_hartree: float
    total_energy_hartree: float
    error_hartree: float
    runtime_seconds: float

@dataclass
class IqpeWorkflowResult:
    """Repeated IQPE results and their unique modal estimate.

    Attributes:
        problem: Shared problem definition used for every repeated run.
        runs: Complete-run results in seed order.
        bitstring_counts: Frequency of each observed complete bitstring.
        modal_run: First run carrying the unique most frequent bitstring.
        total_runtime_seconds: Wall-clock duration of the repeated-run phase.
    """

    problem: IqpeProblem
    runs: list[IqpeRun]
    bitstring_counts: dict[str, int]
    modal_run: IqpeRun
    total_runtime_seconds: float

from tutorial_prepare_trial_state import circuit_statistics
from tutorial_run_iqpe import (
    prepare_iqpe_problem,
    print_iqpe_results,
    run_iqpe_workflow,
)

## Prerequisite concepts

An IQPE calculation requires four kinds of information:

- ***Qubit Hamiltonian***<br>
  A Hermitian operator whose eigenvalue is sought, represented on the compute register.
- ***Trial state***<br>
  A normalized state with nonzero weight on the target Hamiltonian eigenstate.
- ***Time-evolution implementation***<br>
  A logical circuit approximating $e^{-i\hat Ht}$ and controlled powers of that unitary.
- ***Numerical and sampling controls***<br>
  An evolution time, number of phase bits, shots per bit, and number of complete runs.

The previous chapters supplied the qubit Hamiltonian and trial state for the selected molecular problem.
They also recorded the core energy needed to reconstruct the molecular total and the classical CASCI reference used to validate the final estimate; those two quantities are bookkeeping and validation data rather than inputs to the phase-estimation circuit.

The compute register stores the encoded active-space wavefunction.
This chapter adds one readout ancilla, a qubit used to extract phase information without representing another spin orbital.
The [quantum phase estimation overview ↗](https://en.wikipedia.org/wiki/Quantum_phase_estimation_algorithm) provides an optional refresher on the basic controlled-unitary circuit model.

## Energy-to-phase encoding

Let $\vert\Psi_j\rangle$ be an eigenstate of the qubit Hamiltonian with active-space energy $E_j$:

$$
\hat H_{\mathrm{qubit}}\vert\Psi_j\rangle
= E_j\vert\Psi_j\rangle.
$$

In atomic units, the evolution time $t$ is expressed in inverse Hartree, $E_{\mathrm{h}}^{-1}$, so the product $E_jt$ is dimensionless.
The time-evolution unitary is

$$
U(t)=e^{-i\hat H_{\mathrm{qubit}}t}.
$$

Because every power of the Hamiltonian acting on $\vert\Psi_j\rangle$ contributes the corresponding power of $E_j$, the exponential acts on that eigenstate as

$$
U(t)\vert\Psi_j\rangle
=e^{-i\hat H_{\mathrm{qubit}}t}\vert\Psi_j\rangle
=e^{-iE_jt}\vert\Psi_j\rangle.
$$

The physical eigenphase in the exponential is therefore $-E_jt$ modulo $2\pi$.
The phase fraction reported by QPE is

$$
\varphi_j
=\left(\frac{-E_jt}{2\pi}\right)\bmod 1.
$$

The QDK/Chemistry result object handles the modulo wrapping automatically.
It converts the measured phase fraction to a signed angle $\alpha\in(-\pi,\pi]$ and returns $-\alpha/t$.
The tutorial script uses this value directly rather than manually applying a sign conversion.
To avoid aliasing, the active-space Hamiltonian energy eigenvalue $E_j$ being estimated must lie in the signed interval $[-\pi/t,\pi/t)$; energies outside that interval can produce the same measured phase.
The two boundary energies differ by one complete phase turn and therefore represent the same measured phase; QDK/Chemistry assigns that boundary to $-\pi/t$.

The next figure shows how to read this wrapping convention.
Follow the upper axis from $\varphi=0$ toward $\varphi=1$.
From zero through one half, the signed angle is nonnegative, so $E=-\alpha/t$ runs from zero down to $-\pi/t$ along the green branch.
Immediately above one half, the signed angle wraps from $+\pi$ to just above $-\pi$; the corresponding energy jumps to just below $+\pi/t$ and then returns toward zero along the purple branch.
At $\varphi=1/2$, the lower filled point includes $-\pi/t$, while the upper open point excludes the equivalent $+\pi/t$ representation.

<div style="text-align:center;">

<svg xmlns:xlink="http://www.w3.org/1999/xlink" width="656.39952pt" height="339.59952pt" viewBox="0 0 656.39952 339.59952" xmlns="http://www.w3.org/2000/svg" version="1.1" class="qdk-chemistry-phase-wrapping" role="img" data-asset="tutorial_qpe_phase_wrapping.svg" aria-label="Signed reconstructed energy plotted against phase fraction from zero to one. Phase fractions from zero through one half map from zero down to minus pi over t. Immediately above one half, the signed branch wraps to just below plus pi over t and returns toward zero as the phase approaches one. A neutral dashed guide connects the included minus pi over t point and excluded plus pi over t point at phase one half. A red bracket beside the plot spans their energy difference of two pi over t. Phase one wraps to phase zero." style="max-width:100%;height:auto"> <metadata> <rdf:RDF xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:cc="http://creativecommons.org/ns#" xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"> <cc:Work> <dc:type rdf:resource="http://purl.org/dc/dcmitype/StillImage"/> <dc:format>image/svg+xml</dc:format> <dc:creator> <cc:Agent> <dc:title>Matplotlib v3.11.1, https://matplotlib.org/</dc:title> </cc:Agent> </dc:creator> </cc:Work> </rdf:RDF> </metadata> <defs> <style type="text/css"> .qdk-chemistry-phase-wrapping { --qdk-host-foreground: var( --vscode-editor-foreground, var(--jp-widgets-color, currentColor) ); --diagram-foreground: var(--qdk-host-foreground); --diagram-grid: var( --vscode-panel-border, var(--jp-border-color2, #888888) ); } .qdk-chemistry-phase-wrapping text { fill: var(--diagram-foreground) !important; } </style>  <style type="text/css">.qdk-chemistry-phase-wrapping *{stroke-linejoin: round; stroke-linecap: butt}</style> </defs> <g id="figure_1"> <g id="patch_1"> <path d="M 0 339.59952 L 656.39952 339.59952 L 656.39952 0 L 0 0 L 0 339.59952 z " style="fill: none"/> </g> <g id="axes_1"> <g id="patch_2"> <path d="M 64.055469 301.398739 L 649.19952 301.398739 L 649.19952 7.2 L 64.055469 7.2 L 64.055469 301.398739 z " style="fill: none"/> </g> <g id="matplotlib.axis_1"> <g id="xtick_1"> <g id="line2d_1"> <path d="M 74.694452 301.398739 L 74.694452 7.2 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: var(--diagram-grid); stroke-width: 0.8; stroke-linecap: square"/> </g> <g id="line2d_2"> <defs> <path id="mda624babe4" d="M 0 0 L 0 3.5 " style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </defs> <g> <use xlink:href="#mda624babe4" x="74.694452" y="301.398739" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_1"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="74.694452" y="315.996395" transform="rotate(-0 74.694452 315.996395)">0</text> </g> </g> <g id="xtick_2"> <g id="line2d_3"> <path d="M 207.681736 301.398739 L 207.681736 7.2 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: var(--diagram-grid); stroke-width: 0.8; stroke-linecap: square"/> </g> <g id="line2d_4"> <g> <use xlink:href="#mda624babe4" x="207.681736" y="301.398739" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_2"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="207.681736" y="315.996395" transform="rotate(-0 207.681736 315.996395)">1/4</text> </g> </g> <g id="xtick_3"> <g id="line2d_5"> <path d="M 340.66902 301.398739 L 340.66902 7.2 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: var(--diagram-grid); stroke-width: 0.8; stroke-linecap: square"/> </g> <g id="line2d_6"> <g> <use xlink:href="#mda624babe4" x="340.66902" y="301.398739" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_3"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="340.66902" y="315.996395" transform="rotate(-0 340.66902 315.996395)">1/2</text> </g> </g> <g id="xtick_4"> <g id="line2d_7"> <path d="M 473.656305 301.398739 L 473.656305 7.2 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: var(--diagram-grid); stroke-width: 0.8; stroke-linecap: square"/> </g> <g id="line2d_8"> <g> <use xlink:href="#mda624babe4" x="473.656305" y="301.398739" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_4"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="473.656305" y="315.996395" transform="rotate(-0 473.656305 315.996395)">3/4</text> </g> </g> <g id="xtick_5"> <g id="line2d_9"> <path d="M 606.643589 301.398739 L 606.643589 7.2 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: var(--diagram-grid); stroke-width: 0.8; stroke-linecap: square"/> </g> <g id="line2d_10"> <g> <use xlink:href="#mda624babe4" x="606.643589" y="301.398739" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_5"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="606.643589" y="315.996395" transform="rotate(-0 606.643589 315.996395)">1 → 0</text> </g> </g> <g id="text_6"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="356.627494" y="329.997176" transform="rotate(-0 356.627494 329.997176)">Reported phase fraction φ ∈ [0, 1)</text> </g> </g> <g id="matplotlib.axis_2"> <g id="ytick_1"> <g id="line2d_11"> <path d="M 64.055469 29.638887 L 649.19952 29.638887 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: var(--diagram-grid); stroke-width: 0.8; stroke-linecap: square"/> </g> <g id="line2d_12"> <defs> <path id="medb564f5fd" d="M 0 0 L -3.5 0 " style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </defs> <g> <use xlink:href="#medb564f5fd" x="64.055469" y="29.638887" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_7"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end" x="57.055469" y="33.437715" transform="rotate(-0 57.055469 33.437715)">+π/t</text> </g> </g> <g id="ytick_2"> <g id="line2d_13"> <path d="M 64.055469 91.969128 L 649.19952 91.969128 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: var(--diagram-grid); stroke-width: 0.8; stroke-linecap: square"/> </g> <g id="line2d_14"> <g> <use xlink:href="#medb564f5fd" x="64.055469" y="91.969128" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_8"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end" x="57.055469" y="95.767956" transform="rotate(-0 57.055469 95.767956)">+π/(2t)</text> </g> </g> <g id="ytick_3"> <g id="line2d_15"> <path d="M 64.055469 154.299369 L 649.19952 154.299369 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: var(--diagram-grid); stroke-width: 0.8; stroke-linecap: square"/> </g> <g id="line2d_16"> <g> <use xlink:href="#medb564f5fd" x="64.055469" y="154.299369" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_9"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end" x="57.055469" y="158.098197" transform="rotate(-0 57.055469 158.098197)">0</text> </g> </g> <g id="ytick_4"> <g id="line2d_17"> <path d="M 64.055469 216.629611 L 649.19952 216.629611 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: var(--diagram-grid); stroke-width: 0.8; stroke-linecap: square"/> </g> <g id="line2d_18"> <g> <use xlink:href="#medb564f5fd" x="64.055469" y="216.629611" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_10"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end" x="57.055469" y="220.428439" transform="rotate(-0 57.055469 220.428439)">−π/(2t)</text> </g> </g> <g id="ytick_5"> <g id="line2d_19"> <path d="M 64.055469 278.959852 L 649.19952 278.959852 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: var(--diagram-grid); stroke-width: 0.8; stroke-linecap: square"/> </g> <g id="line2d_20"> <g> <use xlink:href="#medb564f5fd" x="64.055469" y="278.959852" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_11"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end" x="57.055469" y="282.75868" transform="rotate(-0 57.055469 282.75868)">−π/t</text> </g> </g> <g id="text_12"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="14.798438" y="154.299369" transform="rotate(-90 14.798438 154.299369)">Signed reconstructed energy</text> </g> </g> <g id="line2d_21"> <path d="M 74.694452 154.299369 L 340.66902 278.959852 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: #00796b; stroke-width: 3; stroke-linecap: square"/> </g> <g id="line2d_22"> <path d="M 340.66902 29.638887 L 606.643589 154.299369 " clip-path="url(#pc895b022ad)" style="fill: none; stroke: #7b1fa2; stroke-width: 3; stroke-linecap: square"/> </g> <g id="line2d_23"> <path d="M 340.66902 301.398739 L 340.66902 7.2 " clip-path="url(#pc895b022ad)" style="fill: none; stroke-dasharray: 5.55,2.4; stroke-dashoffset: 0; stroke: #78909c; stroke-width: 1.5"/> </g> <g id="patch_3"> <path d="M 64.055469 301.398739 L 64.055469 7.2 " style="fill: none; stroke: var(--diagram-foreground); stroke-width: 0.8; stroke-linejoin: miter; stroke-linecap: square"/> </g> <g id="patch_4"> <path d="M 64.055469 301.398739 L 649.19952 301.398739 " style="fill: none; stroke: var(--diagram-foreground); stroke-width: 0.8; stroke-linejoin: miter; stroke-linecap: square"/> </g> <g id="PathCollection_1"> <defs> <path id="m92b30d0e14" d="M 0 4.609772 C 1.222526 4.609772 2.395145 4.124058 3.259601 3.259601 C 4.124058 2.395145 4.609772 1.222526 4.609772 0 C 4.609772 -1.222526 4.124058 -2.395145 3.259601 -3.259601 C 2.395145 -4.124058 1.222526 -4.609772 0 -4.609772 C -1.222526 -4.609772 -2.395145 -4.124058 -3.259601 -3.259601 C -4.124058 -2.395145 -4.609772 -1.222526 -4.609772 0 C -4.609772 1.222526 -4.124058 2.395145 -3.259601 3.259601 C -2.395145 4.124058 -1.222526 4.609772 0 4.609772 z " style="stroke: #00796b"/> </defs> <g clip-path="url(#pc895b022ad)"> <use xlink:href="#m92b30d0e14" x="340.66902" y="278.959852" style="fill: #00796b; stroke: #00796b"/> </g> </g> <g id="PathCollection_2"> <path d="M 340.66902 34.248659 C 341.891546 34.248659 343.064165 33.762944 343.928621 32.898488 C 344.793078 32.034032 345.278792 30.861413 345.278792 29.638887 C 345.278792 28.416361 344.793078 27.243742 343.928621 26.379286 C 343.064165 25.514829 341.891546 25.029115 340.66902 25.029115 C 339.446494 25.029115 338.273875 25.514829 337.409419 26.379286 C 336.544963 27.243742 336.059248 28.416361 336.059248 29.638887 C 336.059248 30.861413 336.544963 32.034032 337.409419 32.898488 C 338.273875 33.762944 339.446494 34.248659 340.66902 34.248659 L 340.66902 34.248659 z " clip-path="url(#pc895b022ad)" style="fill: none; stroke: #7b1fa2; stroke-width: 2.5"/> </g> <g id="text_13"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end; fill: #00796b" x="325.66902" y="296.959852" transform="rotate(-0 325.66902 296.959852)">included boundary</text> </g> <g id="text_14"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: start; fill: #7b1fa2" x="354.66902" y="17.638887" transform="rotate(-0 354.66902 17.638887)">excluded boundary</text> </g> <g id="patch_5"> <path d="M 630.5813 265.311469 Q 630.5813 154.301072 630.5813 43.290675 " style="fill: none; stroke: #c62828; stroke-width: 1.5; stroke-linecap: round"/> <path d="M 632.5813 261.311469 L 630.5813 265.311469 L 628.5813 261.311469 " style="fill: none; stroke: #c62828; stroke-width: 1.5; stroke-linecap: round"/> <path d="M 628.5813 47.290675 L 630.5813 43.290675 L 632.5813 47.290675 " style="fill: none; stroke: #c62828; stroke-width: 1.5; stroke-linecap: round"/> </g> </g> </g> <defs> <clipPath id="pc895b022ad"> <rect x="64.055469" y="7.2" width="585.144051" height="294.198739"/> </clipPath> </defs> </svg>

*QDK/Chemistry converts the wrapped phase fraction to one signed energy branch. The neutral dashed guide marks the shared phase $\varphi=1/2$; the red bracket spans the $2\pi/t$ energy difference between its included and excluded boundary representations. Energies separated by that amount alias to the same reported phase.*

</div>

With $m$ measured phase bits, the representable fractions are multiples of $2^{-m}$, so adjacent energy-grid points are separated by

$$
\Delta E_{\mathrm{grid}}=\frac{2\pi}{t2^m}.
$$

With $m$ measured phase bits, the largest controlled power is $U^{2^{m-1}}$.
Increasing $m$ from six to ten therefore raises the largest power from $U^{32}$ to $U^{512}$, increasing the size and runtime cost of the largest iteration circuit by a factor of sixteen for this repeated-power strategy.
This tutorial uses six bits to keep the simulation tractable; that choice does not by itself provide $\mathrm{m}E_{\mathrm{h}}$ resolution.

The evolution time is computed from quantities already produced by the classically tractable example chosen for this tutorial.
If we write the mapped Hamiltonian as $\hat H_{\mathrm{qubit}}=\sum_\ell h_\ell P_\ell$ and define

$$
\lambda=\sum_\ell\lvert h_\ell\rvert.
$$

The QDK/Chemistry application programming interface (API) exposes this coefficient 1-norm as `qubit_hamiltonian.schatten_norm`; the tutorial script uses it to choose the evolution time and reports it in the pre-simulation settings.
For this Hamiltonian, the reported value is $\lambda=19.610172748837\ E_{\mathrm{h}}$.
Because $\lambda$ bounds the magnitudes of the Hamiltonian eigenvalues, the initial choice

$$
t_{\mathrm{bound}}=\frac{\pi}{\lambda}
=0.160202191680\ E_{\mathrm{h}}^{-1}
$$

keeps the spectrum within the signed, unaliased phase interval.
The script reports this value as the `Initial unaliased time bound` in its pre-simulation settings.
Using the active-space reference from the chapter *Mapping the problem to qubits*, $E_{\mathrm{ref}}=-9.653276065987\ E_{\mathrm{h}}$, this initial time gives the implementation phase fraction

$$
\varphi_{\mathrm{bound}}
=\left(\frac{-t_{\mathrm{bound}}E_{\mathrm{ref}}}{2\pi}\right)\bmod 1
\approx0.246129297014.
$$

The script reports this value as the `Reference phase at initial time bound`.
The nearest six-bit fraction is $16/64=0.25$, represented by `010000`.
Its signed angle is $2\pi(16/64)=+\pi/2$.
Finally, we can choose the evolution time so that this grid point reconstructs an energy $\delta=0.001\ E_{\mathrm{h}}$ above the known reference:

$$
t
=\frac{-\pi/2}{E_{\mathrm{ref}}+\delta}
=0.162738437655\ E_{\mathrm{h}}^{-1}.
$$

The script reports this adjusted value as the `Selected evolution time`.
Using this time, the reference phase fraction is approximately $0.250025901$, only about $2.59\times10^{-5}$ above the selected grid point.
The grid point therefore reconstructs an active energy exactly $1\ \mathrm{m}E_{\mathrm{h}}$ above the classical reference to the displayed precision.

The table below compares the selected point with its neighboring six-bit grid energies and the known reference.
The rows are ordered by energy rather than by grid index, so the more negative $k=17$ energy appears before $k=16$.
The reference row has no grid index or bitstring because the reference does not lie exactly on the six-bit grid.

<div style="display:flex;flex-direction:column;align-items:center;">

| Grid point | Bitstring | Active energy ($E_{\mathrm{h}}$) |
|---|---|---|
| $k=17$ | `010001` | $-10.255543320111$ |
| Known reference (not a grid point) | Not applicable | $-9.653276065987$ |
| $k=16$ (selected) | `010000` | $-9.652276065987$ |
| $k=15$ | `001111` | $-9.049008811863$ |

*Six-bit active-energy grid near the reference. Here $k$ is the integer grid index, so $\varphi_k=k/2^6=k/64$; its six-bit binary representation is the measured bitstring.*

</div>

The neighboring grid energies differ by approximately $0.6033\ E_{\mathrm{h}}$.
By contrast, selected grid point $k=16$ is only $0.001\ E_{\mathrm{h}}$ above the known reference.
This small offset is possible because the evolution time was tuned using that reference; it is not the general resolution of the six-bit grid.

**Please note**: this use of the already known classical energy is circular.
It is useful for this tutorial, but it is not a generally available strategy when the target energy is unknown.
For the chosen $t$, adjacent energies represented by the six-bit phase grid differ by approximately $0.6033\ E_{\mathrm{h}}$, not $0.001\ E_{\mathrm{h}}$.
The smaller value, $0.001\ E_{\mathrm{h}}$ or $1\ \mathrm{m}E_{\mathrm{h}}$, is the accuracy target adopted for this tutorial.
The question below asks why one grid point can nevertheless reconstruct this particular reference energy within that target.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does six-bit phase estimation meet a 1&nbsp;m<i>E</i><sub>h</sub> target here even though adjacent grid points are much farther apart?</summary>

<div style="padding:0.1em 1em;">

The classically known reference energy was used to tune the evolution time so the target lies almost exactly on one six-bit grid point.
Six bits do not provide $\mathrm{m}E_{\mathrm{h}}$ resolution for arbitrary energies with this evolution time.

</div>

</details></div>

## Qubit measurement and shots

Immediately before measurement, the readout ancilla has a state

$$
\vert q\rangle
=\gamma_0\vert0\rangle+\gamma_1\vert1\rangle,
\qquad
|\gamma_0|^2+|\gamma_1|^2=1.
$$

The coefficients $\gamma_0$ and $\gamma_1$ are complex probability amplitudes.
They play the same mathematical role as the determinant coefficients $c_I$ and eigenstate amplitudes $a_j$ introduced earlier, but here they describe the two computational-basis states of one qubit.

Measuring this qubit in the computational basis returns one classical bit.
The probability of measuring zero is $|\gamma_0|^2$, and the probability of measuring one is $|\gamma_1|^2$.
One preparation, circuit execution, and measurement is called a *shot*.

A single shot samples one outcome; it does not reveal the probabilities themselves.
Repeating the same circuit from a freshly prepared state produces counts whose relative frequencies estimate the probabilities.
For example, 70 zero outcomes among 100 shots estimate the probability of zero as $0.70$.

In this IQPE implementation, phase kickback, feedback, and the final H gate make the readout-ancilla probabilities depend on the phase bit being measured.
The circuit uses an odd number of shots and selects the majority outcome as that iteration's bit.
It then uses this classical bit to set the feedback rotation for the next iteration.

The loop executes iteration $k=0$ first.
This iteration applies the largest controlled power and estimates the least-significant phase bit.
Later iterations proceed toward the most-significant bit.

## One phase bit at a time

Standard phase estimation uses several readout ancillas and an inverse quantum Fourier transform to obtain a complete phase in one coherent circuit.
The QDK/Chemistry iterative implementation (IQPE) instead reuses a single readout ancilla across a sequence of independently executed circuits.
This reduces each circuit's logical-qubit requirement, both on quantum hardware and in this tutorial's classical simulator, at the cost of repeated state preparation and circuit execution.
This qubit-resource tradeoff is why the tutorial uses IQPE.

For iterations over $k=0,1,\ldots,m-1$, the circuit builder uses the controlled power

$$
U^{2^{m-k-1}}.
$$

With $m=6$, the six iteration circuits therefore apply powers $32,16,8,4,2,1$.
QDK/Chemistry reverses the measurements from execution order when it constructs the conventional most-significant-bit-first result.
For an input eigenstate, the first H gate prepares the readout ancilla in $(\vert0\rangle+\vert1\rangle)/\sqrt{2}$.
The feedback rotation applies a corrective phase determined by earlier iterations.
The controlled power then produces *phase kickback*: the $\vert1\rangle$ branch acquires the eigenphase of $U^{2^{m-k-1}}$, while the $\vert0\rangle$ branch does not.
Together, the feedback and kickback phases cancel the contribution already determined by earlier iterations.
After the second H gate, the remaining relative phase changes the probabilities of measuring zero or one, revealing the next phase bit.

If iteration $k$ selects bit $b_k$, QDK/Chemistry updates its accumulated feedback angle according to

$$
\Phi_{k+1}=\frac{\Phi_k}{2}+\frac{\pi b_k}{2},
\qquad \Phi_0=0.
$$

After the last iteration, the reported phase fraction is $\Phi_m/\pi$.
The following figure summarizes one iteration, from fresh register preparation through repeated shots, majority voting, and the feedback update for the next phase bit.

<div style="text-align:center;">

<svg width="587pt" height="891pt" viewBox="0.00 0.00 587.00 891.00" xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" role="img" data-asset="tutorial_qpe_iqpe_iteration.svg" aria-label="Flow through one IQPE iteration. In parallel, freshly prepare the molecular trial state on the compute register and a Hadamard superposition on a new readout ancilla. Rotate the ancilla using feedback from earlier measured bits, then use it to control this iteration&#x27;s time-evolution power on the compute register. Apply a final Hadamard gate and measure the ancilla for one shot. Repeat with fresh registers for an odd number of shots; their majority selects phase bit b sub k. Use that bit to update the classical feedback angle and continue to iteration k plus 1." style="max-width:100%;height:auto"> <g id="graph0" class="graph" transform="scale(1 1) rotate(0) translate(14.4 876.6)"> <title>TutorialQpeIqpeIteration</title> <!-- Compute --> <g id="node1" class="node"> <title>Compute</title> <path fill="#e0f2f1" stroke="#00796b" stroke-width="2.5" d="M247.2,-862.2C247.2,-862.2 12,-862.2 12,-862.2 6,-862.2 0,-856.2 0,-850.2 0,-850.2 0,-805.8 0,-805.8 0,-799.8 6,-793.8 12,-793.8 12,-793.8 247.2,-793.8 247.2,-793.8 253.2,-793.8 259.2,-799.8 259.2,-805.8 259.2,-805.8 259.2,-850.2 259.2,-850.2 259.2,-856.2 253.2,-862.2 247.2,-862.2"/> <text xml:space="preserve" text-anchor="start" x="54.22" y="-835.2" font-family="Arial" font-weight="bold" font-size="14.00" fill="#004d40">Fresh compute register</text> <text xml:space="preserve" text-anchor="start" x="54.97" y="-810.05" font-family="Arial" font-size="11.00" fill="#004d40">Prepare the trial state |Ψ</text> <text xml:space="preserve" text-anchor="start" x="173.47" y="-810.05" font-family="Arial" baseline-shift="sub" font-size="11.00" fill="#004d40">trial</text> <text xml:space="preserve" text-anchor="start" x="188.47" y="-810.05" font-family="Arial" font-size="11.00" fill="#004d40">⟩</text> </g> <!-- Evolution --> <g id="node4" class="node"> <title>Evolution</title> <path fill="#e0f2f1" stroke="#00796b" stroke-width="2.5" d="M424.6,-635.4C424.6,-635.4 88.6,-635.4 88.6,-635.4 82.6,-635.4 76.6,-629.4 76.6,-623.4 76.6,-623.4 76.6,-579 76.6,-579 76.6,-573 82.6,-567 88.6,-567 88.6,-567 424.6,-567 424.6,-567 430.6,-567 436.6,-573 436.6,-579 436.6,-579 436.6,-623.4 436.6,-623.4 436.6,-629.4 430.6,-635.4 424.6,-635.4"/> <text xml:space="preserve" text-anchor="start" x="155.35" y="-608.4" font-family="Arial" font-weight="bold" font-size="14.00" fill="#004d40">Apply controlled time evolution</text> <text xml:space="preserve" text-anchor="start" x="124.23" y="-583.25" font-family="Arial" font-size="11.00" fill="#004d40">Ancilla controls U^(2^(m−k−1)) on the compute register</text> </g> <!-- Compute&#45;&gt;Evolution --> <g id="edge2" class="edge"> <title>Compute&#45;&gt;Evolution</title> <path fill="none" stroke="#00796b" stroke-width="2.5" d="M155,-792.56C155,-792.56 155,-649.17 155,-649.17"/> <polygon fill="#00796b" stroke="#00796b" stroke-width="2.5" points="158.15,-649.17 155,-640.17 151.85,-649.17 158.15,-649.17"/> </g> <!-- Ancilla --> <g id="node2" class="node"> <title>Ancilla</title> <path fill="#e8eaf6" stroke="#5c6bc0" stroke-width="2.5" d="M546.2,-862.2C546.2,-862.2 311,-862.2 311,-862.2 305,-862.2 299,-856.2 299,-850.2 299,-850.2 299,-805.8 299,-805.8 299,-799.8 305,-793.8 311,-793.8 311,-793.8 546.2,-793.8 546.2,-793.8 552.2,-793.8 558.2,-799.8 558.2,-805.8 558.2,-805.8 558.2,-850.2 558.2,-850.2 558.2,-856.2 552.2,-862.2 546.2,-862.2"/> <text xml:space="preserve" text-anchor="start" x="343.85" y="-835.2" font-family="Arial" font-weight="bold" font-size="14.00" fill="#283593">Fresh readout ancilla |0⟩</text> <text xml:space="preserve" text-anchor="start" x="378.35" y="-810.05" font-family="Arial" font-size="11.00" fill="#283593">Apply the first H gate</text> </g> <!-- Feedback --> <g id="node3" class="node"> <title>Feedback</title> <path fill="#e8eaf6" stroke="#5c6bc0" stroke-width="2.5" d="M523.8,-748.8C523.8,-748.8 245.4,-748.8 245.4,-748.8 239.4,-748.8 233.4,-742.8 233.4,-736.8 233.4,-736.8 233.4,-692.4 233.4,-692.4 233.4,-686.4 239.4,-680.4 245.4,-680.4 245.4,-680.4 523.8,-680.4 523.8,-680.4 529.8,-680.4 535.8,-686.4 535.8,-692.4 535.8,-692.4 535.8,-736.8 535.8,-736.8 535.8,-742.8 529.8,-748.8 523.8,-748.8"/> <text xml:space="preserve" text-anchor="start" x="313.35" y="-721.8" font-family="Arial" font-weight="bold" font-size="14.00" fill="#283593">Apply phase feedback</text> <text xml:space="preserve" text-anchor="start" x="287.85" y="-696.65" font-family="Arial" font-size="11.00" fill="#283593">Use angle Φ</text> <text xml:space="preserve" text-anchor="start" x="348.6" y="-696.65" font-family="Arial" baseline-shift="sub" font-size="11.00" fill="#283593">k</text> <text xml:space="preserve" text-anchor="start" x="353.1" y="-696.65" font-family="Arial" font-size="11.00" fill="#283593"> from earlier measured bits</text> </g> <!-- Ancilla&#45;&gt;Feedback --> <g id="edge1" class="edge"> <title>Ancilla&#45;&gt;Feedback</title> <path fill="none" stroke="#5c6bc0" stroke-width="2.5" d="M417.4,-792.74C417.4,-792.74 417.4,-762.62 417.4,-762.62"/> <polygon fill="#5c6bc0" stroke="#5c6bc0" stroke-width="2.5" points="420.55,-762.62 417.4,-753.62 414.25,-762.62 420.55,-762.62"/> </g> <!-- Feedback&#45;&gt;Evolution --> <g id="edge3" class="edge"> <title>Feedback&#45;&gt;Evolution</title> <path fill="none" stroke="#5c6bc0" stroke-width="2.5" d="M335,-679.34C335,-679.34 335,-649.22 335,-649.22"/> <polygon fill="#5c6bc0" stroke="#5c6bc0" stroke-width="2.5" points="338.15,-649.22 335,-640.22 331.85,-649.22 338.15,-649.22"/> </g> <!-- Measure --> <g id="node5" class="node"> <title>Measure</title> <path fill="#e8eaf6" stroke="#5c6bc0" stroke-width="2.5" d="M395.8,-522C395.8,-522 117.4,-522 117.4,-522 111.4,-522 105.4,-516 105.4,-510 105.4,-510 105.4,-465.6 105.4,-465.6 105.4,-459.6 111.4,-453.6 117.4,-453.6 117.4,-453.6 395.8,-453.6 395.8,-453.6 401.8,-453.6 407.8,-459.6 407.8,-465.6 407.8,-465.6 407.8,-510 407.8,-510 407.8,-516 401.8,-522 395.8,-522"/> <text xml:space="preserve" text-anchor="start" x="152.35" y="-495" font-family="Arial" font-weight="bold" font-size="14.00" fill="#283593">Apply H and measure the ancilla</text> <text xml:space="preserve" text-anchor="start" x="201.1" y="-469.85" font-family="Arial" font-size="11.00" fill="#283593">One shot returns 0 or 1</text> </g> <!-- Evolution&#45;&gt;Measure --> <g id="edge4" class="edge"> <title>Evolution&#45;&gt;Measure</title> <path fill="none" stroke="#00796b" stroke-width="2.5" d="M256.6,-565.94C256.6,-565.94 256.6,-535.82 256.6,-535.82"/> <polygon fill="#00796b" stroke="#00796b" stroke-width="2.5" points="259.75,-535.82 256.6,-526.82 253.45,-535.82 259.75,-535.82"/> </g> <!-- Shots --> <g id="node6" class="node"> <title>Shots</title> <path fill="#ffffff" stroke="#5c6bc0" stroke-width="2.5" d="M417.4,-408.6C417.4,-408.6 95.8,-408.6 95.8,-408.6 89.8,-408.6 83.8,-402.6 83.8,-396.6 83.8,-396.6 83.8,-352.2 83.8,-352.2 83.8,-346.2 89.8,-340.2 95.8,-340.2 95.8,-340.2 417.4,-340.2 417.4,-340.2 423.4,-340.2 429.4,-346.2 429.4,-352.2 429.4,-352.2 429.4,-396.6 429.4,-396.6 429.4,-402.6 423.4,-408.6 417.4,-408.6"/> <text xml:space="preserve" text-anchor="start" x="133.23" y="-381.6" font-family="Arial" font-weight="bold" font-size="14.00" fill="#3949ab">Repeat with freshly prepared registers</text> <text xml:space="preserve" text-anchor="start" x="138.85" y="-356.45" font-family="Arial" font-size="11.00" fill="#3949ab">Collect an odd number of outcomes for iteration k</text> </g> <!-- Measure&#45;&gt;Shots --> <g id="edge5" class="edge"> <title>Measure&#45;&gt;Shots</title> <path fill="none" stroke="#5c6bc0" stroke-width="2.5" d="M256.6,-452.54C256.6,-452.54 256.6,-422.42 256.6,-422.42"/> <polygon fill="#5c6bc0" stroke="#5c6bc0" stroke-width="2.5" points="259.75,-422.42 256.6,-413.42 253.45,-422.42 259.75,-422.42"/> </g> <!-- Majority --> <g id="node7" class="node"> <title>Majority</title> <path fill="#f3e5f5" stroke="#7b1fa2" stroke-width="2.5" d="M395.8,-295.2C395.8,-295.2 117.4,-295.2 117.4,-295.2 111.4,-295.2 105.4,-289.2 105.4,-283.2 105.4,-283.2 105.4,-238.8 105.4,-238.8 105.4,-232.8 111.4,-226.8 117.4,-226.8 117.4,-226.8 395.8,-226.8 395.8,-226.8 401.8,-226.8 407.8,-232.8 407.8,-238.8 407.8,-238.8 407.8,-283.2 407.8,-283.2 407.8,-289.2 401.8,-295.2 395.8,-295.2"/> <text xml:space="preserve" text-anchor="start" x="170.35" y="-268.2" font-family="Arial" font-weight="bold" font-size="14.00" fill="#4a148c">Majority vote selects bit b</text> <text xml:space="preserve" text-anchor="start" x="336.1" y="-268.2" font-family="Arial" font-weight="bold" baseline-shift="sub" font-size="14.00" fill="#4a148c">k</text> <text xml:space="preserve" text-anchor="start" x="182.35" y="-243.05" font-family="Arial" font-size="11.00" fill="#4a148c">Classical result for this iteration</text> </g> <!-- Shots&#45;&gt;Majority --> <g id="edge6" class="edge"> <title>Shots&#45;&gt;Majority</title> <path fill="none" stroke="#7b1fa2" stroke-width="2.5" d="M256.6,-339.14C256.6,-339.14 256.6,-309.02 256.6,-309.02"/> <polygon fill="#7b1fa2" stroke="#7b1fa2" stroke-width="2.5" points="259.75,-309.02 256.6,-300.02 253.45,-309.02 259.75,-309.02"/> </g> <!-- Update --> <g id="node8" class="node"> <title>Update</title> <path fill="#f3e5f5" stroke="#7b1fa2" stroke-width="2.5" d="M403,-181.8C403,-181.8 110.2,-181.8 110.2,-181.8 104.2,-181.8 98.2,-175.8 98.2,-169.8 98.2,-169.8 98.2,-125.4 98.2,-125.4 98.2,-119.4 104.2,-113.4 110.2,-113.4 110.2,-113.4 403,-113.4 403,-113.4 409,-113.4 415,-119.4 415,-125.4 415,-125.4 415,-169.8 415,-169.8 415,-175.8 409,-181.8 403,-181.8"/> <text xml:space="preserve" text-anchor="start" x="141.1" y="-154.8" font-family="Arial" font-weight="bold" font-size="14.00" fill="#4a148c">Update the classical feedback angle</text> <text xml:space="preserve" text-anchor="start" x="198.48" y="-129.65" font-family="Arial" font-size="11.00" fill="#4a148c">Use b</text> <text xml:space="preserve" text-anchor="start" x="226.98" y="-129.65" font-family="Arial" baseline-shift="sub" font-size="11.00" fill="#4a148c">k</text> <text xml:space="preserve" text-anchor="start" x="231.48" y="-129.65" font-family="Arial" font-size="11.00" fill="#4a148c"> to compute Φ</text> <text xml:space="preserve" text-anchor="start" x="299.73" y="-129.65" font-family="Arial" baseline-shift="sub" font-size="11.00" fill="#4a148c">k+1</text> </g> <!-- Majority&#45;&gt;Update --> <g id="edge7" class="edge"> <title>Majority&#45;&gt;Update</title> <path fill="none" stroke="#7b1fa2" stroke-width="2.5" d="M256.6,-225.74C256.6,-225.74 256.6,-195.62 256.6,-195.62"/> <polygon fill="#7b1fa2" stroke="#7b1fa2" stroke-width="2.5" points="259.75,-195.62 256.6,-186.62 253.45,-195.62 259.75,-195.62"/> </g> <!-- Next --> <g id="node9" class="node"> <title>Next</title> <path fill="#ffffff" stroke="#7b1fa2" stroke-width="2.5" d="M403,-68.4C403,-68.4 110.2,-68.4 110.2,-68.4 104.2,-68.4 98.2,-62.4 98.2,-56.4 98.2,-56.4 98.2,-12 98.2,-12 98.2,-6 104.2,0 110.2,0 110.2,0 403,0 403,0 409,0 415,-6 415,-12 415,-12 415,-56.4 415,-56.4 415,-62.4 409,-68.4 403,-68.4"/> <text xml:space="preserve" text-anchor="start" x="173.35" y="-41.4" font-family="Arial" font-weight="bold" font-size="14.00" fill="#4a148c">Continue to iteration k + 1</text> <text xml:space="preserve" text-anchor="start" x="155.35" y="-16.25" font-family="Arial" font-size="11.00" fill="#4a148c">Build the next controlled power and repeat</text> </g> <!-- Update&#45;&gt;Next --> <g id="edge8" class="edge"> <title>Update&#45;&gt;Next</title> <path fill="none" stroke="#7b1fa2" stroke-width="2.5" d="M256.6,-112.34C256.6,-112.34 256.6,-82.22 256.6,-82.22"/> <polygon fill="#7b1fa2" stroke="#7b1fa2" stroke-width="2.5" points="259.75,-82.22 256.6,-73.22 253.45,-82.22 259.75,-82.22"/> </g> </g> </svg>

*One IQPE iteration estimates phase bit $b_k$. Every shot freshly prepares the trial state and readout ancilla; the majority outcome updates $\Phi_{k+1}=\Phi_k/2+\pi b_k/2$ for the next controlled power.*

</div>

After all iterations, the feedback accumulator determines the final phase fraction.

Each iteration circuit contains twelve compute qubits and one readout ancilla, for thirteen logical qubits in the simulated circuit.
The readout ancilla does not represent an additional molecular spin orbital.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why is trial-state preparation included in every IQPE iteration circuit?</summary>

<div style="padding:0.1em 1em;">

Each phase bit is measured by executing a separate circuit, and every shot begins with newly allocated qubits in the all-zero state.
The state-preparation logical circuit must therefore reload the trial state before each controlled evolution.

</div>

</details></div>

The script configures the native iterative circuit builder and first-order Trotter unitary through nested `AlgorithmRef` objects:

In [ ]:
def iqpe_circuit_builder_reference(
    evolution_time_hartree_inverse: float,
    *,
    num_phase_bits: int = DEFAULT_NUM_PHASE_BITS,
) -> AlgorithmRef:
    """Describe native IQPE with repeated first-order Trotter evolution.

    Args:
        evolution_time_hartree_inverse: Base simulated time in inverse Hartree.
        num_phase_bits: Iteration count and number of reported binary phase places.

    Returns:
        A nested algorithm reference selecting the native iterative builder,
        Pauli-sequence controlled-circuit mapping, one first-order Trotter
        division, and repeated base-unitary powers.
    """
    return AlgorithmRef(
        "qpe_circuit_builder",
        "qdk_iterative",
        num_bits=num_phase_bits,
        controlled_circuit_mapper=AlgorithmRef(
            "controlled_circuit_mapper", "pauli_sequence"
        ),
        unitary_builder=AlgorithmRef(
            "hamiltonian_unitary_builder",
            "trotter",
            time=evolution_time_hartree_inverse,
            order=1,
            num_divisions=1,
            power_strategy="repeat",
        ),
    )

## Numerical controls

Five controls determine the approximation and sampling behavior of this workflow:

- ***Evolution time***<br>
  The value $t$ sets the signed energy interval and spacing of the phase grid, as described above.
  Here it is tuned with the known classical reference to produce a $1\ \mathrm{m}E_{\mathrm{h}}$ grid error.
- **[Hamiltonian simulation ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/algorithms/hamiltonian_unitary_builder.html)**<br>
  The qubit Hamiltonian is a sum of Pauli terms that generally do not commute.
  A first-order [Trotter product formula ↗](https://en.wikipedia.org/wiki/Lie_product_formula) approximates evolution under that sum by applying the exponential of each Pauli term in sequence.
  For $\hat H=\sum_\ell h_\ell P_\ell$, using $r$ Trotter divisions gives

$$
e^{-i\hat Ht}
\approx
\left[\prod_\ell e^{-ih_\ell P_\ell t/r}\right]^r.
$$

  One division ($r=1$) is used for each base evolution in this tutorial.
  Increasing $r$ shortens each simulated time step and generally reduces product-formula error, but repeats the Pauli-term sequence more times and increases circuit cost.
  The repeated-power strategy implemented by QDK/Chemistry constructs each controlled power $U^{2^{m-k-1}}$ by repeating that same approximate base unitary, preserving one consistent approximation across the IQPE iterations.
- ***Phase bits***<br>
  Six bits produce six iteration circuits and $2^6=64$ representable phase fractions.
  Increasing this count refines the grid but causes the largest controlled power, circuit size, and simulator runtime to grow exponentially for the repeated-power strategy.
- ***Shots per bit***<br>
  Each iteration circuit is executed three times.
  The odd shot count prevents a tied bit vote, but finite sampling can still select the less probable bit.
- ***Complete runs***<br>
  The full six-bit procedure is repeated twenty times with simulator seeds 42 through 61.
  Each complete run returns one reconstructed bitstring and energy, and the final estimate uses the most frequent complete bitstring.

The default workflow therefore executes $6\times3\times20=360$ iteration-circuit shots.
Phase-grid error, Trotter error, and sampling variation have different causes and should not be combined with basis-set or active-space model error.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Which control changes energy-grid spacing without changing the molecular Hamiltonian?</summary>

<div style="padding:0.1em 1em;">

The number of phase bits changes how finely the phase interval is discretized.
The evolution time also rescales the grid in energy units, but it simultaneously changes the unaliased energy interval and the simulated evolution.

</div>

</details></div>

## IQPE circuit visualization

The cells below build the six iteration circuits and render the shortest one. No quantum simulation runs here.

<div style="text-align:center;">

<svg xmlns="http://www.w3.org/2000/svg" width="664" height="864" viewBox="0 0 664 864" class="qdk-chemistry-power-one-circuit" role="img" data-asset="tutorial_qpe_power_one_circuit_overview.svg" aria-label="Overview of the power-one IQPE circuit on thirteen wires. The top wire is the readout ancilla and receives an H gate, an Rz feedback rotation labeled zero, the controlled Pauli-evolution block, a final H gate, and measurement and reset. The remaining twelve wires form the compute register. State-preparation blocks act on the compute wires that require preparation operations, the controlled Pauli-evolution block spans the ancilla and compute register, and reset operations return the compute wires to zero after evolution." style="max-width:100%;height:auto"> <style> .qdk-chemistry-power-one-circuit { --qdk-host-foreground: var(--vscode-editor-foreground, var(--jp-widgets-color, currentColor)); --circuit-fg: var(--qdk-host-foreground); --circuit-bg: transparent; --unitary-fill: transparent; --unitary-text: var(--qdk-host-foreground); --measure-fill: #0067b8; --measure-text: #ffffff; --ket-fill: #0067b8; --ket-text: #ffffff; } .qdk-chemistry-power-one-circuit line, .qdk-chemistry-power-one-circuit circle, .qdk-chemistry-power-one-circuit rect { stroke: var(--circuit-fg); stroke-width: 1; } .qdk-chemistry-power-one-circuit text { fill: var(--circuit-fg); dominant-baseline: middle; text-anchor: middle; font-family: "Segoe UI", Arial, sans-serif; } .qdk-chemistry-power-one-circuit .qs-mathtext { font-family: "Cambria Math", "STIX Two Math", serif; font-style: italic; } .qdk-chemistry-power-one-circuit .gate .qs-group-label { fill: var(--circuit-fg); text-anchor: start; } .qdk-chemistry-power-one-circuit .gate-unitary { fill: var(--unitary-fill); } .qdk-chemistry-power-one-circuit .gate text { fill: var(--unitary-text); } .qdk-chemistry-power-one-circuit .control-line, .qdk-chemistry-power-one-circuit .control-dot { fill: var(--circuit-fg); } .qdk-chemistry-power-one-circuit .oplus > line, .qdk-chemistry-power-one-circuit .oplus > circle { fill: var(--circuit-bg); stroke: var(--circuit-fg); stroke-width: 2; } .qdk-chemistry-power-one-circuit .gate-measure { fill: var(--measure-fill); } .qdk-chemistry-power-one-circuit .qs-line-measure, .qdk-chemistry-power-one-circuit .arc-measure { stroke: var(--measure-text); fill: none; } .qdk-chemistry-power-one-circuit .gate-ket { fill: var(--ket-fill); } .qdk-chemistry-power-one-circuit text.ket-text { fill: var(--ket-text); stroke: none; } .qdk-chemistry-power-one-circuit .register-classical { stroke-width: 0.5; } .qdk-chemistry-power-one-circuit .gate-collapse, .qdk-chemistry-power-one-circuit .gate-expand, .qdk-chemistry-power-one-circuit .dropzone-layer { display: none; } </style> <g class="qubit-input-states"><text font-size="16" x="20" y="118" text-anchor="start" dominant-baseline="middle" data-wire="0" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">0</tspan>⟩</text><text font-size="16" x="20" y="222" text-anchor="start" dominant-baseline="middle" data-wire="1" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">1</tspan>⟩</text><text font-size="16" x="20" y="274" text-anchor="start" dominant-baseline="middle" data-wire="2" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">2</tspan>⟩</text><text font-size="16" x="20" y="326" text-anchor="start" dominant-baseline="middle" data-wire="3" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">3</tspan>⟩</text><text font-size="16" x="20" y="378" text-anchor="start" dominant-baseline="middle" data-wire="4" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">4</tspan>⟩</text><text font-size="16" x="20" y="430" text-anchor="start" dominant-baseline="middle" data-wire="5" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">5</tspan>⟩</text><text font-size="16" x="20" y="482" text-anchor="start" dominant-baseline="middle" data-wire="6" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">6</tspan>⟩</text><text font-size="16" x="20" y="534" text-anchor="start" dominant-baseline="middle" data-wire="7" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">7</tspan>⟩</text><text font-size="16" x="20" y="586" text-anchor="start" dominant-baseline="middle" data-wire="8" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">8</tspan>⟩</text><text font-size="16" x="20" y="638" text-anchor="start" dominant-baseline="middle" data-wire="9" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">9</tspan>⟩</text><text font-size="16" x="20" y="690" text-anchor="start" dominant-baseline="middle" data-wire="10" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">10</tspan>⟩</text><text font-size="16" x="20" y="742" text-anchor="start" dominant-baseline="middle" data-wire="11" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">11</tspan>⟩</text><text font-size="16" x="20" y="794" text-anchor="start" dominant-baseline="middle" data-wire="12" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">12</tspan>⟩</text></g><g class="wires"><line x1="40" x2="664" y1="118" y2="118" class="qubit-wire"></line><line x1="40" x2="664" y1="222" y2="222" class="qubit-wire"></line><line x1="40" x2="664" y1="274" y2="274" class="qubit-wire"></line><line x1="40" x2="664" y1="326" y2="326" class="qubit-wire"></line><line x1="40" x2="664" y1="378" y2="378" class="qubit-wire"></line><line x1="40" x2="664" y1="430" y2="430" class="qubit-wire"></line><line x1="40" x2="664" y1="482" y2="482" class="qubit-wire"></line><line x1="40" x2="664" y1="534" y2="534" class="qubit-wire"></line><line x1="40" x2="664" y1="586" y2="586" class="qubit-wire"></line><line x1="40" x2="664" y1="638" y2="638" class="qubit-wire"></line><line x1="40" x2="664" y1="690" y2="690" class="qubit-wire"></line><line x1="40" x2="664" y1="742" y2="742" class="qubit-wire"></line><line x1="40" x2="664" y1="794" y2="794" class="qubit-wire"></line><g><line x1="555" x2="555" y1="118" y2="169" class="register-classical"></line><line x1="553" x2="553" y1="118" y2="171" class="register-classical"></line><line x1="555" x2="664" y1="169" y2="169" class="register-classical"></line><line x1="553" x2="664" y1="171" y2="171" class="register-classical"></line></g></g><g><g class="gate" data-location="0,0" data-expanded="true"><rect class="gate-unitary" x="80" y="46" width="566" height="788" fill-opacity="0" stroke-dasharray="8, 8"></rect><g><g class="gate" data-location="0,0-0,0" data-expanded="true"><rect class="gate-unitary" x="90" y="72" width="546" height="752" fill-opacity="0" stroke-dasharray="8, 8"></rect><g><g class="gate" data-location="0,0-0,0-0,0"><g><line x1="165.5" x2="165.5" y1="222" y2="742" stroke-dasharray="8, 8"></line><g><rect class="gate-unitary" x="100" y="202" width="131" height="248" data-wire-ys="[222,274,326,378,430]" data-width="131"></rect><text font-size="14" x="165.5" y="326" class="qs-maintext"><tspan class="qs-mathtext">StatePreparation</tspan></text></g><g><rect class="gate-unitary" x="100" y="514" width="131" height="248" data-wire-ys="[534,586,638,690,742]" data-width="131"></rect><text font-size="14" x="165.5" y="638" class="qs-maintext"><tspan class="qs-mathtext">StatePreparation</tspan></text></g></g><g class="gate-control gate-expand"><circle cx="102" cy="204" r="10"></circle><path d="M102,197 v14 M95,204 h14"></path></g></g><g class="gate" data-location="0,0-0,0-0,1"><g><g><rect class="gate-unitary" x="145.5" y="98" width="40" height="40" data-wire-ys="[118]" data-width="40"></rect><text font-size="14" x="165.5" y="118" class="qs-maintext"><tspan class="qs-mathtext">H</tspan></text></g></g></g><g class="gate" data-location="0,0-0,0-1,0"><g><g><rect class="gate-unitary" x="243" y="98" width="56" height="40" data-wire-ys="[118]" data-width="56"></rect><text font-size="14" x="271" y="111" class="qs-maintext"><tspan class="qs-mathtext">Rz</tspan></text><text font-size="12" x="271" y="126" class="arg-button">0.0000</text></g></g></g><g class="gate" data-location="0,0-0,0-2,0"><g><g><rect class="gate-unitary" x="311" y="98" width="159" height="716" data-wire-ys="[118,222,274,326,378,430,482,534,586,638,690,742,794]" data-width="159"></rect><text font-size="14" x="390.5" y="456" class="qs-maintext"><tspan class="qs-mathtext">RepControlledPauliExp</tspan></text></g></g><g class="gate-control gate-expand"><circle cx="313" cy="100" r="10"></circle><path d="M313,93 v14 M306,100 h14"></path></g></g><g class="gate" data-location="0,0-0,0-3,0"><g><g><rect class="gate-unitary" x="482" y="98" width="40" height="40" data-wire-ys="[118]" data-width="40"></rect><text font-size="14" x="502" y="118" class="qs-maintext"><tspan class="qs-mathtext">H</tspan></text></g></g></g><g class="gate" data-location="0,0-0,0-3,1"><g><g><rect class="gate-ket" x="482" y="202" width="40" height="40" data-wire-ys="[222]" data-width="40"></rect><text font-size="14" x="502" y="222" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-3,2"><g><g><rect class="gate-ket" x="482" y="254" width="40" height="40" data-wire-ys="[274]" data-width="40"></rect><text font-size="14" x="502" y="274" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-3,3"><g><g><rect class="gate-ket" x="482" y="306" width="40" height="40" data-wire-ys="[326]" data-width="40"></rect><text font-size="14" x="502" y="326" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-3,4"><g><g><rect class="gate-ket" x="482" y="358" width="40" height="40" data-wire-ys="[378]" data-width="40"></rect><text font-size="14" x="502" y="378" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-3,5"><g><g><rect class="gate-ket" x="482" y="410" width="40" height="40" data-wire-ys="[430]" data-width="40"></rect><text font-size="14" x="502" y="430" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-3,6"><g><g><rect class="gate-ket" x="482" y="462" width="40" height="40" data-wire-ys="[482]" data-width="40"></rect><text font-size="14" x="502" y="482" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-3,7"><g><g><rect class="gate-ket" x="482" y="514" width="40" height="40" data-wire-ys="[534]" data-width="40"></rect><text font-size="14" x="502" y="534" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-3,8"><g><g><rect class="gate-ket" x="482" y="566" width="40" height="40" data-wire-ys="[586]" data-width="40"></rect><text font-size="14" x="502" y="586" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-3,9"><g><g><rect class="gate-ket" x="482" y="618" width="40" height="40" data-wire-ys="[638]" data-width="40"></rect><text font-size="14" x="502" y="638" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-3,10"><g><g><rect class="gate-ket" x="482" y="670" width="40" height="40" data-wire-ys="[690]" data-width="40"></rect><text font-size="14" x="502" y="690" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-3,11"><g><g><rect class="gate-ket" x="482" y="722" width="40" height="40" data-wire-ys="[742]" data-width="40"></rect><text font-size="14" x="502" y="742" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-3,12"><g><g><rect class="gate-ket" x="482" y="774" width="40" height="40" data-wire-ys="[794]" data-width="40"></rect><text font-size="14" x="502" y="794" class="qs-maintext ket-text">|0⟩</text></g></g></g><g class="gate" data-location="0,0-0,0-4,0"><g><rect class="gate-measure" x="534" y="98" width="40" height="40" data-wire-ys="[118]" data-width="40"></rect><path class="arc-measure" d="M 569 120 A 15 12 0 0 0 539 120" style="pointer-events: none;"></path><line x1="554" x2="566" y1="126" y2="106" class="qs-line-measure" style="pointer-events: none;"></line></g></g><g class="gate" data-location="0,0-0,0-5,0"><g><g><rect class="gate-ket" x="586" y="98" width="40" height="40" data-wire-ys="[118]" data-width="40"></rect><text font-size="14" x="606" y="118" class="qs-maintext ket-text">|0⟩</text></g></g></g></g><text font-size="14" x="100" y="87" class="qs-maintext qs-group-label"><tspan class="qs-mathtext">RunIQPE</tspan></text><g class="gate-control gate-collapse"><circle cx="92" cy="74" r="10"></circle><path d="M85,74 h14"></path></g></g></g><text font-size="14" x="90" y="61" class="qs-maintext qs-group-label"><tspan class="qs-mathtext">MakeIQPECircuit</tspan></text><g class="gate-control gate-collapse"><circle cx="82" cy="48" r="10"></circle><path d="M75,48 h14"></path></g></g></g> </svg>

*Overview of the rendered power-one iteration circuit. Dashed outlines mark the nested `MakeIQPECircuit` and `RunIQPE` Q# operations; the solid boxes show their principal composite operations.*

</div>

The top wire, $\lvert\psi_0\rangle$, is readout ancilla q0.
Its first H gate creates a superposition, and the `Rz(0.0000)` block applies the phase-feedback rotation.
This static preview constructs all six circuits with the builder's initial feedback angle of zero, so the displayed rotation is zero.
During an actual IQPE run, each iteration circuit is rebuilt using the accumulated feedback from earlier measured bits; the power-one iteration can therefore have a nonzero feedback rotation.

The lower wires, $\lvert\psi_1\rangle$ through $\lvert\psi_{12}\rangle$, are compute-register qubits q1–q12.
The `StatePreparation` blocks load the four-determinant trial state on the subsets of compute wires that require preparation operations; blank wires remain part of the compute register.
The `RepControlledPauliExp` block is the power-one controlled first-order Trotter evolution.
The ancilla controls this block, and the resulting phase kickback places the Hamiltonian eigenphase on the ancilla's relative phase.
The final H gate converts that relative phase into measurement probabilities, the measurement produces one shot outcome, and the blue reset operations return the allocated qubits to $\lvert0\rangle$.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;How can you identify the readout ancilla in the rendered circuit?</summary>

<div style="padding:0.1em 1em;">

The q0 wire receives the H gates and feedback rotation, controls the Hamiltonian evolution, and is measured to obtain the phase bit.
The other twelve wires hold the prepared molecular state and form the compute register.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why do all six iteration circuits have the same width but different lengths?</summary>

<div style="padding:0.1em 1em;">

Every iteration uses the same twelve-qubit compute register and one readout ancilla, so each circuit has thirteen logical qubits.
Different controlled powers repeat the approximate time-evolution unitary different numbers of times, changing the logical gate count rather than the register size.

</div>

</details></div>

In [ ]:
from IPython.display import Markdown, display
from qdk.widgets import Circuit

problem = prepare_iqpe_problem()

circuit_powers = [32, 16, 8, 4, 2, 1]

circuit_rows = [
    (power, *circuit_statistics(circuit)[:2])
    for power, circuit in zip(
        circuit_powers, problem.iteration_circuits, strict=True
    )
]

table_rows = [
    "| Controlled power | Logical qubits | Decomposed logical gates |",
    "|---:|---:|---:|",
]

for power, qubits, gates in circuit_rows:
    table_rows.append(f"| {power} | {qubits} | {gates:,} |")

display(Markdown("\n".join(table_rows)))

shortest_circuit = problem.iteration_circuits[-1]
num_qubits, num_gates, gate_counts = circuit_statistics(shortest_circuit)

## Register roles

The circuit viewer numbers wires from top to bottom:

- **q0** is the readout ancilla. Its H gates, feedback rotation, controlled evolution, and measurement extract one phase bit.
- **q1 to q12** are the compute register. They hold the encoded molecular trial state and receive the controlled Hamiltonian evolution.

The readout ancilla is algorithm workspace; it is not a thirteenth molecular spin orbital.

## Render the shortest circuit

The complete decomposed circuit is long because it contains one state preparation and one controlled first-order Trotter sequence for a 247-term Hamiltonian. Use the viewer's pan and zoom controls to inspect the ancilla operations and the controlled gates connecting q0 to the compute register.

In [ ]:
display(Circuit(shortest_circuit.get_qsharp_circuit()))

## Validate the circuit structure

Before trusting any phase-estimation result, confirm the circuit is the one you think it is. Three numbers describe it, and each is somewhere learners commonly go wrong.

Complete `validate_circuit` so it reports:

- `iteration_circuits`, how many iteration circuits one complete run uses
- `compute_qubits`, how many wires hold the encoded molecular state
- `readout_ancillas`, how many wires are algorithm workspace rather than spin orbitals

Read each one from `problem` or from the statistics computed above, rather than typing the numbers in. The stub below gets all three wrong.

In [ ]:
from _unit import exercise


@exercise
def validate_circuit():
    return {
        "iteration_circuits": 1,
        "compute_qubits": num_qubits,
        "readout_ancillas": 0,
    }

**Hint**

`problem.iteration_circuits` is a list, which settles the first one. `problem.mapping` knows how wide the compute register is. For the third, compare that width with `num_qubits`, the width of the circuit you rendered, and re-read the note above about what the extra wire is.

In [ ]:
@exercise
def validate_circuit():
    compute = problem.mapping.num_compute_qubits
    return {
        "iteration_circuits": len(problem.iteration_circuits),
        "compute_qubits": compute,
        "readout_ancillas": num_qubits - compute,
    }

Six, twelve, and one. The rendered circuit is thirteen wires wide, but only twelve carry the molecule: two per active spatial orbital. The thirteenth is the readout ancilla, and it is rebuilt along with the trial state on every one of the six iterations, which is why the preparation cost from *Preparing the Trial State* is paid six times over.

## Iterative phase estimation

The script prepares the mapped Hamiltonian and four-determinant trial state, applies the reference-guided evolution-time choice, and constructs all six iteration circuits before starting any simulation.
This separation lets you inspect settings and circuit counts without accidentally repeating the expensive calculation.

The shortest iteration circuit uses controlled $U$ rather than a higher repeated power.
It still contains the state-preparation operations, twelve-wire compute register, one-wire readout ancilla, feedback rotation, controlled Trotter evolution, and ancilla measurement shared by every iteration.
The companion Jupyter notebook introduced below renders this circuit without simulating it.

One complete IQPE run then invokes the native phase-estimation algorithm with one simulator seed:

In [ ]:
def run_complete_iqpe(problem: IqpeProblem, *, seed: int) -> IqpeRun:
    """Execute one complete bitwise IQPE run with a reproducible simulator seed.

    Args:
        problem: Shared Hamiltonian, trial state, phase controls, and builder setup.
        seed: Full-state simulator seed controlling all finite-shot measurements.

    Returns:
        Canonical bitstring, reconstructed active/total energies, reference error,
        and simulator runtime for one complete phase estimate.
    """
    iqpe = create(
        "phase_estimation",
        "qdk_iterative",
        shots_per_bit=problem.shots_per_bit,
    )
    iqpe.settings().set("qpe_circuit_builder", problem.circuit_builder_reference)
    iqpe.settings().set(
        "circuit_executor",
        AlgorithmRef("circuit_executor", "qdk_full_state_simulator", seed=seed),
    )

    start = perf_counter()
    result = iqpe.run(
        state_preparation=problem.trial_state.circuit,
        qubit_hamiltonian=problem.mapping.qubit_hamiltonian,
    )
    runtime_seconds = perf_counter() - start

    # IQPE measures bits in implementation iteration order. Reconstruct the grid
    # integer from the final phase fraction, then format conventional MSB-first
    # binary so bitstring positions have their familiar place values.
    grid_size = 2**problem.num_phase_bits
    grid_index = round(result.phase_fraction * grid_size) % grid_size
    bitstring = f"{grid_index:0{problem.num_phase_bits}b}"

    # Phase estimation returns the active qubit-Hamiltonian eigenvalue. Add the
    # nuclear/frozen-orbital core term omitted during mapping, then compare with
    # CASCI for exactly the same selected Hamiltonian.
    total_energy = result.raw_energy + problem.mapping.core_energy
    energy_error = total_energy - problem.mapping.active_space_result.refined_energy
    return IqpeRun(
        seed=seed,
        bitstring=bitstring,
        phase_fraction=result.phase_fraction,
        active_energy_hartree=result.raw_energy,
        total_energy_hartree=total_energy,
        error_hartree=energy_error,
        runtime_seconds=runtime_seconds,
    )

Each iteration contributes one measured phase bit to the complete IQPE result.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;How does IQPE use the result from each iteration to construct the final bitstring and phase fraction?</summary>

<div style="padding:0.1em 1em;">

The majority measurement for one iteration selects a phase bit, which updates the classical phase feedback used by the next iteration.
After all six iterations, the feedback calculation combines the measured bits into one phase fraction.
The script writes that fraction as a conventional six-bit string, with the most significant bit first.

</div>

</details></div>

## Repeated complete runs

One complete run can differ from another because every phase bit is selected from a finite number of simulator shots and the trial state contains several Hamiltonian eigenstates.
The workflow therefore repeats the complete six-bit procedure with twenty deterministic simulator seeds.

The final aggregation rule selects the most frequent complete bitstring, or *mode*.
This differs from the majority vote used inside one complete run: a per-bit majority chooses one bit from three shots, whereas the complete-run mode chooses one reconstructed bitstring from twenty runs.
If several bitstrings tie for the highest count, the script reports that no unique mode exists instead of silently choosing one.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why should the final aggregation use complete bitstrings rather than vote on each bit across complete runs?</summary>

<div style="padding:0.1em 1em;">

Each complete bitstring represents one phase-grid point and its corresponding energy.
Voting independently on bits could assemble a bitstring that was never produced by any complete run and would discard the observed joint distribution.

</div>

</details></div>

## Molecular energy reconstruction

After the repeated complete runs, the script selects the bitstring observed most often.
Interpret this bitstring as a binary integer $b$.
If the calculation measures $m$ phase bits, convert $b$ to the phase fraction

$$
\varphi=\frac{b}{2^m}.
$$

QDK/Chemistry converts $2\pi\varphi$ to its equivalent signed angle $\alpha$ between $-\pi$ and $\pi$.
Negating that angle and dividing by the evolution time maps the measured phase to the active-space energy:

$$
E_{\mathrm{active}}^{\mathrm{IQPE}}=\frac{-\alpha}{t}.
$$

This estimates an eigenvalue of the qubit Hamiltonian, not yet the selected-space molecular total.
Finite phase resolution, sampling, and product-formula time evolution all contribute error.
As the chapter *Mapping the problem to qubits* explains, the mapper does not include the core energy in the qubit Hamiltonian.
The core energy contains the nuclear repulsion and the constant contribution from frozen inactive orbitals.
Because these contributions are not measured by phase estimation, the script adds them classically:

$$
E_{\mathrm{total}}^{\mathrm{IQPE}}
=E_{\mathrm{active}}^{\mathrm{IQPE}}+E_{\mathrm{core}}.
$$

Finally, compare this reconstructed total with the CASCI energy of the same selected-space Hamiltonian:

$$
\Delta E_{\mathrm{algorithm}}
=E_{\mathrm{total}}^{\mathrm{IQPE}}-E_{\mathrm{CASCI}}.
$$

The workflow meets the teaching target when $\lvert\Delta E_{\mathrm{algorithm}}\rvert\leq1\ \mathrm{m}E_{\mathrm{h}}$.
This comparison evaluates the configured quantum algorithm against its classical reference; it does not measure basis-set or active-space model error.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Which energy comparison determines whether the IQPE workflow meets the teaching target?</summary>

<div style="padding:0.1em 1em;">

Compare the reconstructed IQPE total energy with the CASCI energy of the same selected active-space Hamiltonian.
Comparing with experiment or a larger orbital space would mix algorithmic error with model error.

</div>

</details></div>

## Read a phase off the grid

Each complete run returns a six-bit string. That string is the binary representation of a grid index $k$, and the estimated phase is $\varphi_k = k / 2^{6}$.

Complete `measured_phase` so it converts the measured bitstring into its phase fraction.

In [ ]:
from _unit import exercise

measured_bitstring = "010000"


@exercise
def measured_phase():
    return 0.0

**Hint**

Two steps. First turn the bitstring into the grid index it represents, which `int` can do if you tell it the base. Then scale that index by the size of the grid, which follows from how many bits the string has.

In [ ]:
@exercise
def measured_phase():
    return int(measured_bitstring, 2) / 2 ** len(measured_bitstring)

The selected grid point `010000` is $k=16$, so $\varphi = 16/64 = 0.25$.

## The complete workflow

The cell below runs the complete workflow. It is the long step in this chapter and reports progress after each complete run.

The script prints its settings before simulation and reports progress for every complete run, including the seed, bitstring, total energy, error, and elapsed time.
A successful run completes all twenty runs and prints the complete-run bitstring counts, most frequent bitstring, component energies, reconstructed total, reference energy, and signed error.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;What bitstring distribution and energy estimate did the script produce?</summary>

<div style="padding:0.1em 1em;">

The bitstring `010000` appeared 19 times and `001111` appeared once, so `010000` was the most frequent result.
It produced an active-space energy of $-9.652276065987\ E_{\mathrm{h}}$ and a reconstructed total of $-108.770051792909\ E_{\mathrm{h}}$ after adding the core energy.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Does the result meet the teaching target, and what does that establish?</summary>

<div style="padding:0.1em 1em;">

The reconstructed total is $+1\ \mathrm{m}E_{\mathrm{h}}$ above the selected-space CASCI reference, meeting the teaching target at its boundary.
That offset was deliberately set by the reference-guided phase-grid alignment, while Trotter approximation and finite sampling can still affect which bitstring is selected.
This classical simulation of the quantum calculation therefore validates this configured teaching workflow; it does not remove molecular-model error or establish agreement with experiment.

</div>

</details></div>

In [ ]:
iqpe_result = run_iqpe_workflow()
print_iqpe_results(iqpe_result)

## Knowledge check

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;What changes if the number of phase bits increases while the repeated-power strategy remains fixed?</summary>

<div style="padding:0.1em 1em;">

The phase grid becomes finer, but an additional iteration circuit is required and the largest controlled-unitary power doubles.
For repeated-power Trotter evolution, that larger power increases circuit size and simulator runtime substantially.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Would increasing shots per bit make the phase grid finer?</summary>

<div style="padding:0.1em 1em;">

No.
More shots can make each bit majority more stable, but grid spacing is controlled by the evolution time and number of phase bits.

</div>

</details></div>

## What you accomplished

You completed an end-to-end molecular-energy workflow: defining stretched N<sub>2</sub> in an orbital basis, selecting an active space, mapping its Hamiltonian to qubits, preparing a multiconfigurational trial state, simulating iterative phase estimation, and reconstructing the molecular total energy by adding the core energy.

The final comparison shows that this configured IQPE workflow reproduces the matching selected-space CASCI reference within the tutorial's $1\ \mathrm{m}E_{\mathrm{h}}$ target.

A useful next investigation would change one layer at a time: enlarge the molecular model, choose a different trial state, or vary the phase-estimation controls, then identify which accuracy and cost measures respond.

## Further reading

- [Phase estimation ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/algorithms/phase_estimation.html)
- [Phase-estimation circuit builders ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/algorithms/qpe_circuit_builder.html)
- [Hamiltonian unitary builders ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/algorithms/hamiltonian_unitary_builder.html)
- [Circuit execution ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/algorithms/circuit_executor.html)
- [Phase-estimation results ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/data/qpe_result.html)